In [ ]:
#@title LocalFold { display-mode: "form" }
import json, os, queue, subprocess, sys, threading, time

BRANCH, PORT, REPO = 'main', 8710, '/content/localfold'
# 🔴 chrome IS THE DEFAULT BECAUSE IT IS THE ONE THAT FOLDS EVERYTHING. The
# node runtime is this process over Dawn - no browser, 6.7 s of cold setup
# against 15.4 - and it re-implements the fold path rather than driving the
# page, so measured on a T4 it folds ONE PROTEIN CHAIN: multi-chain, copies,
# ligands, SMILES, contacts and modified residues are all refused, a DNA chain
# comes back as a PEPTIDE, no alignment is ever searched for and AlphaFold 2 is
# refused by name. chrome opens index.html?role=runtime and presses its
# controls, so it inherits every one of those for free. See docs/DAWN.md.
RUNTIME = 'chrome'

# The setup: a Vulkan-capable driver, node with Dawn, and this repository.
# Each step checks the disk first, so re-running the cell is nearly free. The
# reasoning behind the unusual choices here (the driver unpacked rather than
# installed, Dawn rather than a browser) is in docs/WEB.md.
SETUP = r"""
set -e
say() { echo "[$(( $(date +%s) - T0 ))s] $*"; }
T0=$(date +%s)

# What is missing? Each probe is an `if`: under `set -e` a bare `[ -e X ] && f=0`
# would end the script when the test fails, which is the case we are detecting.
need_icd=1
if [ -e /usr/share/vulkan/icd.d/nvidia_icd.json ]; then need_icd=0; fi
need_node=1
if [ -x /content/node-v24.8.0-linux-x64/bin/node ]; then need_node=0; fi
need_loader=1
if ldconfig -p 2>/dev/null | grep -q libvulkan.so.1; then need_loader=0; fi
# ...and a browser only when one is going to be driven. WANT_CHROME is RUNTIME
# above; the zip is 261 MB and 13 s, which the node runtime has no use for.
need_chrome=0
if [ "WANT_CHROME" = 1 ] && ! command -v google-chrome > /dev/null 2>&1; then
  need_chrome=1
fi
say "icd=$need_icd node=$need_node loader=$need_loader chrome=$need_chrome"

# 🔴 CHROME IS A ZIP, NOT A .deb - AND `chrome-headless-shell`, WHICH IS THE
# HALF WE USE. Measured on a T4: the .deb is 0.65 s to fetch and 21.4 s in
# dpkg, while Chrome for Testing is 1.8 s down and 11 s to unzip, with no
# package manager and no triggers. Both report nvidia / turing / shader-f16
# through a served page: a zip is not a lesser browser, it is the same binary
# without dpkg. 261 MB against 393.
if [ "$need_chrome" = 1 ]; then
  ( cd /content \
    && wget -q https://storage.googleapis.com/chrome-for-testing-public/153.0.8010.52/linux64/chrome-headless-shell-linux64.zip \
    && unzip -q -o chrome-headless-shell-linux64.zip -d /opt \
    && ln -sf /opt/chrome-headless-shell-linux64/chrome-headless-shell /usr/local/bin/google-chrome ) &
  chrome_download=$!
fi

# Node and Dawn, which is what folds. Dawn is Chrome's own WebGPU
# implementation as an npm package: 6.7-6.9 s here against 15.4-16.7 for a
# headless Chrome, measured cold on two fresh T4s, and no X11 libraries
# because nothing is drawing. Node 24 because the model scales are float16 and
# `Float16Array` is absent on Colab's own node 20 and on 22.14.
if [ "$need_node" = 1 ]; then
  ( cd /content \
    && wget -q https://nodejs.org/dist/v24.8.0/node-v24.8.0-linux-x64.tar.xz \
    && tar xf node-v24.8.0-linux-x64.tar.xz \
    && ln -sf /content/node-v24.8.0-linux-x64/bin/node /usr/local/bin/node \
    && ln -sf /content/node-v24.8.0-linux-x64/bin/npm /usr/local/bin/npm ) &
  node_download=$!
fi

# The repository. Started first and waited for last: it is git's network while
# the rest is dpkg's lock, so it costs nothing to run underneath them.
if [ -d REPO_PATH ]; then
  ( git -C REPO_PATH fetch -q origin BRANCH && git -C REPO_PATH checkout -q FETCH_HEAD ) &
else
  ( git clone -q --depth 1 --branch BRANCH https://github.com/sokrypton/localfold REPO_PATH ) &
fi
repo_clone=$!

# The Vulkan loader. The four X11 libraries that used to be here - libatk,
# libatk-bridge, libatspi and libxcomposite - were a zipped Chrome's alone,
# 6.0 s of apt for a browser that is no longer installed.
if [ "$need_loader" = 1 ] || [ "$need_icd" = 1 ] || [ "$need_chrome" = 1 ]; then
  APT="apt-get -qq -o Dpkg::Use-Pty=0 --no-install-recommends"
  # The Vulkan loader, and - only for a browser - the four libraries a zipped
  # Chrome asks for that this image lacks; `ldd` named exactly these. The `t64`
  # spelling is Ubuntu 24.04's and the bare one 22.04's. They are a browser's
  # alone, which is why they are behind `need_chrome`.
  X11=""
  if [ "$need_chrome" = 1 ]; then
    X11="libatk1.0-0t64 libatk-bridge2.0-0t64 libatspi2.0-0t64 libxcomposite1"
    X11_OLD="libatk1.0-0 libatk-bridge2.0-0 libatspi2.0-0 libxcomposite1"
  fi
  ( $APT install -y libvulkan1 $X11 > /dev/null 2>&1 \
    || $APT install -y libvulkan1 $X11_OLD > /dev/null 2>&1 \
    || { apt-get -qq update > /dev/null 2>&1
         $APT install -y libvulkan1 $X11 > /dev/null 2>&1 \
           || $APT install -y libvulkan1 $X11_OLD > /dev/null 2>&1; } ) &
  small_libs=$!
fi

# Colab already has the NVIDIA userspace; what it lacks is a libGLX_nvidia
# carrying the Vulkan entry point, without which WebGPU finds no adapter. Only
# that package is unpacked, into `/` so the linker mixes it with Colab's own
# libraries at the kernel module's version.
if [ "$need_icd" = 1 ]; then
  D=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | cut -d. -f1)
  if [ -n "$D" ]; then
    ( cd /content && apt-get -qq download libnvidia-gl-$D > /dev/null 2>&1 \
      && dpkg-deb -x libnvidia-gl-${D}_*.deb / && ldconfig )
    say "driver (unpacked, not installed)"
  fi
fi
# 🔴 `wait $x` WITH `x` UNSET IS A BARE `wait`, WHICH REAPS EVERY JOB - and a
# second wait on a reaped pid returns 127, which `set -e` turns into an exit
# with no message. On a WARM box need_loader and need_icd are both 0, so
# `small_libs` is never set, the bare wait collects the clone too, and
# `wait $repo_clone` below then ends the setup at rc 127 with the last `say`
# unprinted. Cold boxes passed and every re-run failed - the case this
# script's own header calls "nearly free", and the one its own failure
# message sends the reader back into. Measured on a T4.
if [ -n "$small_libs" ]; then wait $small_libs 2>/dev/null; fi
say "vulkan"

if [ "$need_node" = 1 ]; then
  wait $node_download
  say "node $(node --version 2>/dev/null || echo 'DID NOT INSTALL')"
fi

if [ "$need_chrome" = 1 ]; then
  wait $chrome_download
  say "chrome $(google-chrome --version 2>/dev/null || echo 'DID NOT INSTALL')"
fi

wait $repo_clone 2>/dev/null
say "repository"

# Dawn itself, into the repository so the runtime's own import finds it. An
# OPTIONAL peer dependency rather than a dependency, which is why it is
# installed here and not named in package.json: the page needs none of it.
# 🔴 PINNED. webgpu 0.6.0's Dawn ABORTS this fold on a Colab T4 - the trunk
# finishes and the process dies of SIGABRT the moment diffusion starts, with
# no uncaptured error and no lost device, so nothing reaches JavaScript to be
# reported. 0.4.0 folds the same sequence to completion on the same machine:
# pLDDT 70.1, backbone 1.45/1.54/3.86 A. Measured both ways in one session.
( cd REPO_PATH && npm i webgpu@0.4.0 --silent --no-audit --no-fund > /dev/null 2>&1 )
say "dawn"
"""

with open('/content/_localfold_setup.sh', 'w') as handle:
    handle.write(SETUP.replace('REPO_PATH', REPO).replace('BRANCH', BRANCH)
                      .replace('WANT_CHROME', '1' if RUNTIME == 'chrome' else '0'))

# The log is kept, not printed: it is only worth reading when there is no
# button at the end, so it is shown in full if the setup fails.
print('setting up…')
_t0 = time.time()
_setup = subprocess.run(['bash', '/content/_localfold_setup.sh'],
                        capture_output=True, text=True)
if _setup.returncode != 0:
    print(_setup.stdout, _setup.stderr)
    raise SystemExit('the setup failed; the log is above')
_t_setup = time.time() - _t0

# The service both serves the page and brokers for it, which is what makes one
# cell enough: the link below is index.html on this machine, so the page and
# the GPU are the same origin. It opens a second, headless copy of that page on
# the card, and the two talk through web/colab-bridge.js — yours asks, that one
# folds and pushes back every status line and sampler frame as it happens.
def _drain(stream, sink):
    for line in iter(stream.readline, ''):
        sink.put(line.rstrip())

if not globals().get('_localfold_service'):
    # The runtime's address is passed explicitly rather than inherited:
    # Disconnect hands the machine back with a POST to $TBE_RUNTIME_ADDR, so a
    # subprocess that merely hopes to inherit it would stop the service and
    # leave the machine assigned.
    _env = dict(os.environ)
    if 'TBE_RUNTIME_ADDR' in os.environ:
        _env['TBE_RUNTIME_ADDR'] = os.environ['TBE_RUNTIME_ADDR']
    _localfold_service = subprocess.Popen(
        [sys.executable, 'tools/colab_backend.py', '--port', str(PORT),
         '--runtime', RUNTIME],
        cwd=REPO, env=_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    _lines = queue.Queue()
    threading.Thread(target=_drain, args=(_localfold_service.stdout, _lines), daemon=True).start()
    TOKEN, ADAPTER = None, None
    _deadline = time.time() + 300
    while time.time() < _deadline and TOKEN is None:
        try:
            said = _lines.get(timeout=5)
        except queue.Empty:
            continue
        if said.startswith('BACKEND '):
            answer = json.loads(said[len('BACKEND '):])
            TOKEN, ADAPTER = answer['token'], answer['gpu']
        elif 'rror' in said:
            print(said)
    if TOKEN is None:
        raise SystemExit('the fold service did not start; run this cell again')

    # Belt to the service's braces: whichever notices first hands the machine
    # back, and the second call is a no-op. A thread rather than a loop, so the
    # cell finishes and the link stays clickable.
    def _release_when_done(service):
        service.wait()
        try:
            from google.colab import runtime as _runtime
            _runtime.unassign()
            print('the Colab runtime has been released')
            return
        except Exception as cause:
            print(f'unassign did not take ({cause}); disconnecting the'
                  ' frontend instead')
        # Colab's Runtime menu is not reachable from here — it lives in the
        # notebook page's DOM, a different origin from an output frame. A
        # machine nobody is connected to is reclaimed on idle.
        try:
            from google.colab import output as _out
            _out.eval_js('google.colab.kernel.disconnect()', ignore_result=True)
            print('the notebook has been disconnected; the machine idles out')
        except Exception as cause:
            print('the fold service stopped; the runtime is still assigned'
                  f' ({cause}) - use Runtime > Disconnect and delete runtime')

    threading.Thread(target=_release_when_done, args=(_localfold_service,),
                     daemon=True).start()

# 'nvidia' and an architecture is the card. 'swiftshader' or 'llvmpipe' is the
# CPU wearing its clothes, and every fold after that looks exactly like success
# — so that case is loud and the good case is four words.
_card = '%s %s' % (ADAPTER.get('vendor') or '?', ADAPTER.get('architecture') or '')
_fallback = ('swiftshader' in json.dumps(ADAPTER).lower()
             or 'llvmpipe' in json.dumps(ADAPTER).lower())
if _fallback:
    print('*** %s is the CPU renderer, not the card.' % _card,
          'Runtime > Change runtime type > T4 GPU, then run this cell again. ***')

from google.colab.output import eval_js
from IPython.display import HTML, display

base = eval_js('google.colab.kernel.proxyPort(%d)' % PORT)
# The base may or may not end in a slash. Without this, '%sindex.html' once
# produced '…prod.colab.devindex.html', which the browser reads as a hostname.
base = base if base.endswith('/') else base + '/'
# `backend=colab` tells the page which machine folds; `t` is the token every
# request carries, in the URL because a page cannot be handed a header by
# whoever opened it.
page = '%sindex.html?backend=colab&t=%s' % (base, TOKEN)
display(HTML(
    '<div style="font:14px system-ui;padding:14px 16px;border:1px solid #e5e7eb;'
    'border-radius:10px;display:inline-block">'
    '<a href="%s" target="_blank" rel="noopener" '
    'style="font-size:17px;font-weight:600;text-decoration:none">LocalFold &rarr;</a>'
    '<div style="color:#6b7280;margin-top:6px">'
    'keep this notebook running</div>'
    '</div>' % page))
